# Giai đoạn 2 – Làm sạch & chuẩn hóa dữ liệu (Data Wrangling)
**Dự án:** A Day in the Life of a Data Analyst – VTI Academy

**Đầu vào** (để cùng thư mục với notebook):
- `Final Project.csv` – khảo sát Stack Overflow (11.552 dòng, 85 cột)
- `jobs_raw.csv`, `job_skills.csv` – tin tuyển dụng đã cào từ 123job.vn

**Đầu ra:**
- `clean_data.xlsx` – **1 file duy nhất** gồm 5 sheet đã làm sạch (nộp cho chị)
- thư mục `clean_csv/` – cùng dữ liệu dạng CSV (để nạp vào SQL Server)

**Các bước:** (1) làm sạch khảo sát → (2) tách kỹ năng sang bảng dài → (3) làm sạch tin tuyển dụng → (4) kiểm tra → (5) lưu file.

## 0. Import thư viện và khai báo tên file

In [1]:
import glob
import re
import os
import sys
import subprocess
import importlib
import numpy as np
import pandas as pd

def ensure(package):
    """Cài thư viện nếu máy chưa có (tránh dừng giữa chừng ở bước lưu Excel)."""
    try:
        importlib.import_module(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

ensure("xlsxwriter")   # thư viện ghi file Excel nhanh cho bảng lớn

# Tìm file khảo sát theo mẫu tên (chấp nhận "Final Project.csv" hoặc "Final_Project.csv")
SURVEY_FILE = glob.glob("Final*Project*.csv")[0]
JOBS_FILE = "jobs_raw.csv"
JOB_SKILLS_FILE = "job_skills.csv"

OUT_XLSX = "clean_data.xlsx"   # file nộp
OUT_CSV_DIR = "clean_csv"      # bản CSV để nạp SQL
print("Khảo sát:", SURVEY_FILE)

Khảo sát: Final Project.csv


## 1. Làm sạch khảo sát Stack Overflow
### 1.1 Đọc dữ liệu và bỏ dòng trùng

In [2]:
survey = pd.read_csv(SURVEY_FILE)
print("Ban đầu:", survey.shape)

# Bỏ các dòng trùng hoàn toàn (file gốc có 154 dòng trùng)
n_before = len(survey)
survey = survey.drop_duplicates().reset_index(drop=True)
print(f"Đã bỏ {n_before - len(survey)} dòng trùng -> còn {len(survey)} dòng")

# Mỗi người trả lời phải xuất hiện đúng 1 lần
assert survey["Respondent"].is_unique, "Còn Respondent bị trùng!"

Ban đầu: (11552, 85)
Đã bỏ 154 dòng trùng -> còn 11398 dòng


### 1.2 Chuyển cột "số nhưng lẫn chữ" thành số
`YearsCode`, `YearsCodePro` có giá trị `Less than 1 year` / `More than 50 years`; `Age1stCode` có `Younger than 5 years` / `Older than 85`. Ép về số để tính toán được.

In [3]:
def to_number(series, text_map):
    """Đổi các giá trị chữ đặc biệt theo text_map, rồi ép cả cột về kiểu số."""
    return pd.to_numeric(series.replace(text_map), errors="coerce")

survey["YearsCode"] = to_number(survey["YearsCode"],
    {"Less than 1 year": 0, "More than 50 years": 51})
survey["YearsCodePro"] = to_number(survey["YearsCodePro"],
    {"Less than 1 year": 0, "More than 50 years": 51})
survey["Age1stCode"] = to_number(survey["Age1stCode"],
    {"Younger than 5 years": 4, "Older than 85": 86})

### 1.3 Xử lý giá trị bất thường (outlier)
Quy tắc (ghi rõ để trả lời Q&A): giá trị **không thể có thật** hoặc vô lý được đặt thành thiếu (NaN), không xóa cả dòng để giữ lại các câu trả lời khác của người đó.
- `Age`: hợp lệ trong khoảng 15–80
- `WorkWeekHrs`: tối đa 100 giờ/tuần (file có giá trị tới 1012)
- `ConvertedComp` (lương năm quy đổi USD): bỏ giá trị dưới 1.000 USD (không thực tế cho lương cả năm, gồm cả giá trị 0) và trên 1.000.000 USD

In [4]:
n_age = ((survey["Age"] < 15) | (survey["Age"] > 80)).sum()
n_hrs = (survey["WorkWeekHrs"] > 100).sum()
n_comp = ((survey["ConvertedComp"] < 1000) | (survey["ConvertedComp"] > 1_000_000)).sum()

survey.loc[(survey["Age"] < 15) | (survey["Age"] > 80), "Age"] = np.nan
survey.loc[survey["WorkWeekHrs"] > 100, "WorkWeekHrs"] = np.nan
survey.loc[(survey["ConvertedComp"] < 1000) | (survey["ConvertedComp"] > 1_000_000), "ConvertedComp"] = np.nan
print(f"Đặt NaN: Age {n_age}, WorkWeekHrs {n_hrs}, ConvertedComp {n_comp}")

Đặt NaN: Age 1, WorkWeekHrs 88, ConvertedComp 315


### 1.4 Chuẩn hóa giá trị chữ và thêm cột phân nhóm
- Tên nước: `Viet Nam` → `Vietnam`
- Giới tính (cột chọn nhiều): gom về `Man`, `Woman`, `Non-binary/Other`, `Unknown`
- Nhóm kinh nghiệm chuyên nghiệp (`exp_group`) để chia theo cấp độ khi phân tích
- Cột phân loại còn thiếu → `Unknown` (để slicer Power BI không hiện `(Blank)`)

In [5]:
survey["Country"] = survey["Country"].replace({"Viet Nam": "Vietnam"})

def clean_gender(x):
    """Gom cột giới tính (có thể chọn nhiều, ngăn cách bằng ';') về 4 nhóm."""
    if pd.isna(x):
        return "Unknown"
    parts = {p.strip() for p in x.split(";")}
    if parts == {"Man"}:
        return "Man"
    if parts == {"Woman"}:
        return "Woman"
    return "Non-binary/Other"

survey["Gender"] = survey["Gender"].apply(clean_gender)

# Nhóm số năm viết code chuyên nghiệp
bins = [-0.1, 2, 5, 10, 20, 100]
labels = ["0-2 nam", "3-5 nam", "6-10 nam", "11-20 nam", "Tren 20 nam"]
survey["ExpGroup"] = pd.cut(survey["YearsCodePro"], bins=bins, labels=labels).astype(object)
survey["ExpGroup"] = survey["ExpGroup"].fillna("Unknown")

survey["IsVietnam"] = (survey["Country"] == "Vietnam").astype(int)

# Cột phân loại: thiếu -> "Unknown"
cat_cols = ["MainBranch", "Employment", "Country", "Student", "EdLevel", "UndergradMajor",
            "OrgSize", "DevType", "JobSeek", "OpSys", "WorkRemote", "JobSat", "CareerSat", "Hobbyist"]
for c in cat_cols:
    survey[c] = survey[c].fillna("Unknown")

### 1.5 Chọn cột cần dùng và đổi tên sang `snake_case`
Đề yêu cầu tên cột rõ ràng, không khoảng trắng, không ký tự đặc biệt. Chỉ giữ cột phục vụ phân tích (bỏ các cột như `BlockchainOrg`, `SOVisitFreq`... không liên quan đến bài toán kỹ năng). Thêm các cột `answered_*` (1 nếu người đó có trả lời câu hỏi tương ứng) để làm **mẫu số** khi tính tỷ lệ %.

In [6]:
COLUMN_RENAME = {
    "Respondent": "respondent_id",
    "MainBranch": "main_branch",
    "Hobbyist": "hobbyist",
    "Employment": "employment",
    "Country": "country",
    "IsVietnam": "is_vietnam",
    "Student": "student",
    "EdLevel": "education_level",
    "UndergradMajor": "undergrad_major",
    "OrgSize": "org_size",
    "DevType": "dev_type",
    "YearsCode": "years_code",
    "Age1stCode": "age_first_code",
    "YearsCodePro": "years_code_pro",
    "ExpGroup": "exp_group",
    "CareerSat": "career_satisfaction",
    "JobSat": "job_satisfaction",
    "JobSeek": "job_seek",
    "WorkWeekHrs": "work_week_hours",
    "ConvertedComp": "annual_comp_usd",
    "WorkRemote": "work_remote",
    "OpSys": "operating_system",
    "Age": "age",
    "Gender": "gender",
}
survey_clean = survey[list(COLUMN_RENAME)].rename(columns=COLUMN_RENAME)

# Cờ "đã trả lời câu hỏi" (1/0) - dùng làm MẪU SỐ khi tính % kỹ năng.
# Lý do: không phải ai cũng trả lời câu "muốn dùng năm tới"; nếu chia cho tổng số người
# thì % bị thấp đi, nhất là với framework và misc tech (chỉ khoảng 86-87% có trả lời).
ANSWER_FLAGS = {
    "answered_language_worked":  ["LanguageWorkedWith"],
    "answered_language_desire":  ["LanguageDesireNextYear"],
    "answered_database_worked":  ["DatabaseWorkedWith"],
    "answered_database_desire":  ["DatabaseDesireNextYear"],
    "answered_framework_worked": ["WebFrameWorkedWith", "MiscTechWorkedWith"],   # trả lời 1 trong 2 là tính
    "answered_framework_desire": ["WebFrameDesireNextYear", "MiscTechDesireNextYear"],
    "answered_platform_worked":  ["PlatformWorkedWith"],
    "answered_platform_desire":  ["PlatformDesireNextYear"],
    "answered_ide":              ["DevEnviron"],
}
for flag, source_cols in ANSWER_FLAGS.items():
    survey_clean[flag] = survey[source_cols].notna().any(axis=1).astype(int)

survey_clean.head()

,respondent_id,main_branch,hobbyist,employment,country,is_vietnam,student,education_level,undergrad_major,org_size,...,gender,answered_language_worked,answered_language_desire,answered_database_worked,answered_database_desire,answered_framework_worked,answered_framework_desire,answered_platform_worked,answered_platform_desire,answered_ide
0,4,I am a developer by profession,No,Employed full-time,United States,0,No,"Bachelor’s degree (BA, BS, B.Eng., etc.)","Computer science, computer engineering, or sof...",100 to 499 employees,...,Man,1,1,1,1,1,1,1,1,1
1,9,I am a developer by profession,Yes,Employed full-time,New Zealand,0,No,Some college/university study without earning ...,"Computer science, computer engineering, or sof...",10 to 19 employees,...,Man,1,1,1,1,1,1,1,1,1
2,13,I am a developer by profession,Yes,Employed full-time,United States,0,No,"Master’s degree (MA, MS, M.Eng., MBA, etc.)","Computer science, computer engineering, or sof...",10 to 19 employees,...,Man,1,1,1,1,1,1,1,1,1
3,16,I am a developer by profession,Yes,Employed full-time,United Kingdom,0,No,"Master’s degree (MA, MS, M.Eng., MBA, etc.)",Unknown,100 to 499 employees,...,Man,1,1,1,1,1,1,0,1,1
4,17,I am a developer by profession,Yes,Employed full-time,Australia,0,No,"Bachelor’s degree (BA, BS, B.Eng., etc.)","Computer science, computer engineering, or sof...",20 to 99 employees,...,Man,1,1,1,1,1,1,1,1,1


## 2. Tách kỹ năng của khảo sát sang bảng dài (long format)
Các cột như `LanguageWorkedWith` chứa nhiều giá trị trong 1 ô (`Python;SQL;Java`). Ta tách thành **mỗi dòng = 1 người × 1 kỹ năng × 1 trạng thái** để đếm tần suất dễ dàng (cả trong SQL lẫn Power BI).

- `skill_group`: `language`, `database`, `framework` (gồm web framework và misc tech), `platform_tool`, `ide`
- `status`: `worked_with` (đã dùng năm qua) hoặc `desire_next_year` (muốn dùng năm tới)

**Chuẩn hóa tên kỹ năng** để khớp với tin tuyển dụng (ví dụ `Microsoft SQL Server` → `SQL Server`, `React.js` → `React`). Giá trị `Other(s):` bị bỏ vì không biết là kỹ năng nào.

In [7]:
# (cột trong khảo sát, nhóm kỹ năng, trạng thái)
SKILL_COLUMNS = [
    ("LanguageWorkedWith",     "language",      "worked_with"),
    ("LanguageDesireNextYear", "language",      "desire_next_year"),
    ("DatabaseWorkedWith",     "database",      "worked_with"),
    ("DatabaseDesireNextYear", "database",      "desire_next_year"),
    ("WebFrameWorkedWith",     "framework",     "worked_with"),
    ("WebFrameDesireNextYear", "framework",     "desire_next_year"),
    ("MiscTechWorkedWith",     "framework",     "worked_with"),
    ("MiscTechDesireNextYear", "framework",     "desire_next_year"),
    ("PlatformWorkedWith",     "platform_tool", "worked_with"),
    ("PlatformDesireNextYear", "platform_tool", "desire_next_year"),
    ("DevEnviron",             "ide",           "worked_with"),   # khảo sát không hỏi IDE muốn dùng
]

# Bảng đổi tên: tên trong khảo sát -> tên chuẩn dùng chung với tin tuyển dụng
SKILL_RENAME = {
    "Bash/Shell/PowerShell": "Bash/Shell",
    "Microsoft SQL Server": "SQL Server",
    "Microsoft Azure": "Azure",
    "Google Cloud Platform": "Google Cloud",
    "React.js": "React",
    "Angular/Angular.js": "Angular",
    "Torch/PyTorch": "PyTorch",
    "IPython / Jupyter": "Jupyter",
}

parts = []
for col, group, status in SKILL_COLUMNS:
    tmp = survey[["Respondent", col]].dropna().copy()
    tmp["skill"] = tmp[col].str.split(";")          # tách chuỗi theo dấu ';'
    tmp = tmp.explode("skill")                      # mỗi kỹ năng 1 dòng
    tmp["skill"] = tmp["skill"].str.strip()
    tmp = tmp[tmp["skill"] != "Other(s):"]          # bỏ giá trị không xác định
    tmp["skill"] = tmp["skill"].replace(SKILL_RENAME)
    tmp["skill_group"] = group
    tmp["status"] = status
    tmp["source_field"] = col                       # nhớ lại cột gốc để truy vết
    parts.append(tmp[["Respondent", "skill_group", "skill", "status", "source_field"]])

survey_skills = pd.concat(parts, ignore_index=True).rename(columns={"Respondent": "respondent_id"})
# Một người có thể chọn cùng 1 kỹ năng ở cả web framework và misc tech -> bỏ trùng
survey_skills = survey_skills.drop_duplicates(["respondent_id", "skill_group", "skill", "status"]).reset_index(drop=True)
print("Bảng kỹ năng khảo sát:", survey_skills.shape)
survey_skills.groupby(["skill_group", "status"]).size()

Bảng kỹ năng khảo sát: (382659, 5)


skill_group    status          
database       desire_next_year    28930
               worked_with         30647
framework      desire_next_year    50166
               worked_with         42844
ide            worked_with         33306
language       desire_next_year    54928
               worked_with         58952
platform_tool  desire_next_year    44007
               worked_with         38879
dtype: int64

## 3. Làm sạch tin tuyển dụng (123job.vn)
### 3.1 Đọc dữ liệu, bỏ tin trùng và tin không có thông tin

In [8]:
jobs = pd.read_csv(JOBS_FILE)
job_skills_raw = pd.read_csv(JOB_SKILLS_FILE)
print("Ban đầu:", jobs.shape)

# Bỏ trùng theo URL, rồi theo (tiêu đề, công ty, địa điểm, độ dài nội dung) - tin đăng lại
jobs = jobs.drop_duplicates("job_url")
jobs = jobs.drop_duplicates(["job_title", "company", "location", "description_length"])

# Tin không có mô tả (độ dài 0) VÀ cũng không rút được kỹ năng nào từ tiêu đề -> không có thông tin, bỏ.
# Tin rỗng mô tả nhưng tiêu đề có kỹ năng (vd "Lập trình viên Java") thì GIỮ.
urls_with_skill = set(job_skills_raw["job_url"])
empty_no_skill = (jobs["description_length"] == 0) & ~jobs["job_url"].isin(urls_with_skill)
print(f"Bỏ {empty_no_skill.sum()} tin không có thông tin; giữ lại "
      f"{((jobs['description_length'] == 0) & ~empty_no_skill).sum()} tin rỗng mô tả nhưng có kỹ năng ở tiêu đề")
jobs = jobs[~empty_no_skill].reset_index(drop=True)
print(f"Còn {len(jobs)} tin")

Ban đầu: (428, 10)
Bỏ 2 tin không có thông tin; giữ lại 0 tin rỗng mô tả nhưng có kỹ năng ở tiêu đề
Còn 425 tin


### 3.2 Chuẩn hóa văn bản, địa điểm và tạo mã tin
- Công ty: bỏ khoảng trắng thừa, đổi về dạng `Title Case` (nguồn lúc viết hoa lúc viết thường), rồi viết hoa lại các từ viết tắt (`TNHH`, `FPT`...)
- Địa điểm: lấy **tỉnh/thành** (phần cuối của chuỗi như `Quận 10, Hồ Chí Minh, Quận 10, Hồ Chí Minh`)
- `job_id`: lấy từ **đuôi của URL** (vd `...-o9jyvd8PDR` → `o9jyvd8PDR`), nên không đổi khi cào lại; làm khóa nối với bảng kỹ năng
- `has_any_skill`: tin có nhắc ít nhất 1 kỹ năng trong từ điển hay không (dùng làm mẫu số khi tính %)

In [9]:
def squeeze(s):
    """Bỏ khoảng trắng thừa ở đầu/cuối và giữa chuỗi."""
    return s.astype("string").str.strip().str.replace(r"\s+", " ", regex=True)

jobs["job_title"] = squeeze(jobs["job_title"])
ACRONYM_RX = re.compile(r"\b(Tnhh|Mtv|Jsc|Fpt|Vng|Tma|Vn|It|Cp|Sg|Hcm)\b")
def fix_company(name):
    """Title Case rồi viết hoa lại các từ viết tắt thường gặp."""
    if pd.isna(name):
        return "Unknown"
    return ACRONYM_RX.sub(lambda m: m.group(1).upper(), name.title())
jobs["company"] = squeeze(jobs["company"]).apply(fix_company)
jobs["province"] = squeeze(jobs["location"]).str.split(",").str[-1].str.strip().fillna("Unknown")
jobs["industry"] = squeeze(jobs["industry"]).fillna("Unknown")

# Mã tin: đuôi URL (ổn định khi cào lại, khác với số thứ tự)
jobs["job_id"] = jobs["job_url"].str.extract(r"-([A-Za-z0-9]+)$")[0]
assert jobs["job_id"].notna().all() and jobs["job_id"].is_unique, "Không rút được job_id duy nhất từ URL"

# Mã số 1, 2, 3... cho mỗi tin: dùng làm khóa nối trong SQL/Power BI
# (job_id phân biệt hoa/thường, SQL Server và Power BI lại không phân biệt -> báo trùng)
jobs["job_key"] = range(1, len(jobs) + 1)

skill_cols = ["language_list", "database_list", "framework_list", "platform_tool_list"]
jobs[skill_cols] = jobs[skill_cols].fillna("")
jobs["has_any_skill"] = (jobs[skill_cols] != "").any(axis=1).astype(int)

job_postings = jobs[["job_key", "job_id", "job_title", "company", "province", "industry",
                     "description_length", "has_any_skill", "job_url"]]
print(job_postings["province"].value_counts().head(8))

province
Hồ Chí Minh    168
Hà Nội         159
Unknown         49
Hải Phòng        8
Cần Thơ          5
Đà Nẵng          5
Bắc Ninh         4
Điện Biên        3
Name: count, dtype: int64


### 3.3 Bảng kỹ năng của tin tuyển dụng
Giữ các tin còn lại, gắn `job_id`. Đổi nhãn nhóm `skill_group` đã trùng tên với khảo sát (`language`, `database`, `framework`, `platform_tool`); thêm cột `status = required_in_job` để gộp được với khảo sát khi cần.

In [10]:
url_to_key = dict(zip(jobs["job_url"], jobs["job_key"]))
url_to_id  = dict(zip(jobs["job_url"], jobs["job_id"]))
job_skills = job_skills_raw[job_skills_raw["job_url"].isin(url_to_key)].copy()
job_skills["job_key"] = job_skills["job_url"].map(url_to_key)
job_skills["job_id"]  = job_skills["job_url"].map(url_to_id)
job_skills["status"] = "required_in_job"
job_skills = (job_skills[["job_key", "job_id", "skill_group", "skill", "status"]]
              .drop_duplicates().reset_index(drop=True))
print("Bảng kỹ năng tin tuyển dụng:", job_skills.shape)


Bảng kỹ năng tin tuyển dụng: (1523, 5)


## 4. Kiểm tra chất lượng sau khi làm sạch

In [11]:
# Khóa duy nhất, nối bảng không bị "mồ côi"
assert survey_clean["respondent_id"].is_unique
assert job_postings["job_id"].is_unique
assert job_postings["job_key"].is_unique
assert job_skills["job_key"].isin(job_postings["job_key"]).all()
assert survey_skills["respondent_id"].isin(survey_clean["respondent_id"]).all()
assert job_skills["job_id"].isin(job_postings["job_id"]).all()

# Ai có dòng kỹ năng thì phải có cờ "đã trả lời" tương ứng = 1
flag_check = {("language", "worked_with"): "answered_language_worked", ("language", "desire_next_year"): "answered_language_desire",
              ("database", "worked_with"): "answered_database_worked", ("database", "desire_next_year"): "answered_database_desire",
              ("framework", "worked_with"): "answered_framework_worked", ("framework", "desire_next_year"): "answered_framework_desire",
              ("platform_tool", "worked_with"): "answered_platform_worked", ("platform_tool", "desire_next_year"): "answered_platform_desire",
              ("ide", "worked_with"): "answered_ide"}
flags = survey_clean.set_index("respondent_id")
for (grp, st), flag in flag_check.items():
    ids = survey_skills[(survey_skills["skill_group"] == grp) & (survey_skills["status"] == st)]["respondent_id"].unique()
    assert (flags.loc[ids, flag] == 1).all(), flag

# Tên cột: chỉ gồm chữ thường, số và dấu gạch dưới
for name, d in [("survey_respondents", survey_clean), ("survey_skills", survey_skills),
                ("job_postings", job_postings), ("job_skills", job_skills)]:
    assert all(re.fullmatch(r"[a-z0-9_]+", c) for c in d.columns), name
print("Kiểm tra khóa và tên cột: OK\n")

print("Số bản ghi mỗi nguồn:")
print(f"  Khảo sát Stack Overflow : {len(survey_clean):>7,} người trả lời")
print(f"  Tin tuyển dụng 123job   : {len(job_postings):>7,} tin")
print(f"  Tổng                    : {len(survey_clean) + len(job_postings):>7,} bản ghi")

print("\nTỷ lệ thiếu (%) của các cột số chính:")
print(survey_clean[["age", "years_code_pro", "work_week_hours", "annual_comp_usd"]].isna().mean().mul(100).round(1))
print("\nTin tuyển dụng có nhắc kỹ năng:", f"{job_postings['has_any_skill'].mean():.0%}")

Kiểm tra khóa và tên cột: OK

Số bản ghi mỗi nguồn:
  Khảo sát Stack Overflow :  11,398 người trả lời
  Tin tuyển dụng 123job   :     425 tin
  Tổng                    :  11,823 bản ghi

Tỷ lệ thiếu (%) của các cột số chính:
age                2.5
years_code_pro     0.1
work_week_hours    1.8
annual_comp_usd    9.9
dtype: float64

Tin tuyển dụng có nhắc kỹ năng: 77%


## 5. Lưu kết quả
- `clean_data.xlsx`: **1 file, 5 sheet** (nộp cho chị, nạp vào Power BI)
- `clean_csv/*.csv`: bản CSV để nạp vào SQL Server bằng SSMS (mã hóa UTF-8)

In [12]:
readme = pd.DataFrame({
    "sheet": ["survey_respondents", "survey_skills", "job_postings", "job_skills",
              "LUU_Y_mau_so", "LUU_Y_nodejs", "LUU_Y_ide", "LUU_Y_mau"],
    "mo_ta": [
        "1 dòng = 1 người trả lời khảo sát Stack Overflow (đã bỏ trùng, xử lý outlier, đổi tên cột)",
        "1 dòng = 1 người x 1 kỹ năng x 1 trạng thái (worked_with / desire_next_year)",
        "1 dòng = 1 tin tuyển dụng từ 123job.vn (đã chuẩn hóa công ty, tỉnh/thành)",
        "1 dòng = 1 tin x 1 kỹ năng nhắc đến trong tin (status = required_in_job)",
        "Khi tính % trong khảo sát: chia cho số người đã trả lời câu đó (cột answered_* trong survey_respondents), KHÔNG chia cho tổng 11.398 người.",
        "Trong tin tuyển dụng, Node.js được tính vào JavaScript; trong khảo sát Node.js nằm ở nhóm framework. Khi so sánh hai nguồn nên bỏ Node.js hoặc ghi chú rõ.",
        "IDE chỉ có trong khảo sát (tin tuyển dụng hầu như không nhắc IDE).",
        "Kỹ năng trong tin tuyển dụng được rút bằng so khớp từ khóa (gần đúng); mẫu 123job nhỏ, so sánh hai nguồn theo thứ hạng thay vì giá trị tuyệt đối.",
    ],
    "khoa_noi": [
        "respondent_id", "respondent_id -> survey_respondents", "job_key", "job_key -> job_postings",
        "", "", "", "",
    ],
})

os.makedirs(OUT_CSV_DIR, exist_ok=True)
tables = {
    "readme": readme,
    "survey_respondents": survey_clean,
    "survey_skills": survey_skills,
    "job_postings": job_postings,
    "job_skills": job_skills,
}

# File Excel duy nhất (engine xlsxwriter ghi nhanh với bảng lớn)
with pd.ExcelWriter(OUT_XLSX, engine="xlsxwriter") as writer:
    for name, d in tables.items():
        d.to_excel(writer, sheet_name=name, index=False)
        ws = writer.sheets[name]
        ws.freeze_panes(1, 0)                       # cố định hàng tiêu đề
        ws.set_column(0, len(d.columns) - 1, 18)    # độ rộng cột

# Bản CSV (utf-8-sig để Excel/SSMS đọc đúng tiếng Việt)
for name, d in tables.items():
    if name != "readme":
        d.to_csv(f"{OUT_CSV_DIR}/{name}.csv", index=False, encoding="utf-8-sig")

print("Đã lưu:", OUT_XLSX, "và thư mục", OUT_CSV_DIR)

Đã lưu: clean_data.xlsx và thư mục clean_csv
